our first step in preparing the data would be to merge the datasets , adding a new column called "new" signaling if a car is new(yes) or used(no) at the same time , we're gonna choose which columns to keep , process which helped decide the ones to keep was the previous crisp phase , main reasons for eliminations were not enough data in the features , or insiginoficant/irrelevant data ..etc



we'll obviously keep pricing, brand , model , kilometrage(which is gonna be 0 for new cars) , body type , fuel type , and seats(theres doors but it's kinda redundant with this) , transmission, the new flag which will be added , i've decided to ditch puissance en chevaux cause it's lacking a lot in used cars ,instead i'll be using puissance fiscale , ill be keeping date mise en circulation (which will be 2025 for the new cars ) , i'll be keeping climatisation(auto or manual) , car dimensions will be ditched as thats already incorporated in body type , fuel usage and everything motor related is also ditched cause ill be keeping puissance fiscale which is directly linked with those .
in short here's how it looks : 

- Price (convert new cars to numeric)
- Brand
- Model
- Kilométrage (0 for new cars)
- Body type (Carrosserie)
- Fuel type (Energie)
- Seats (Nombre de places)
- Transmission (Boîte/Boite vitesse)
- Puissance fiscale (fiscal power)
- Date mise en circulation (2025 for new cars)
- Climatisation (auto/manual)
- New flag (yes/no)



In [79]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data
df_new = pd.read_csv("../data_scraper/data_scraped/automobileTnNew.csv")
df_used = pd.read_csv("../data_scraper/data_scraped/automobileTnUsed.csv")

def clean_year_string(year_val):
    if pd.isna(year_val):
        return None
    # Convert to string and handle float representation
    year_str = str(year_val)
    # If it looks like a float (has dot and digits), fix it
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            month = parts[0]
            year_part = parts[1]
            if len(year_part) == 3:
                year_part = year_part + '0' 
            return f"{month}.{year_part}"
    return year_str

df_used['Mise en circulation'] = df_used['Mise en circulation'].apply(clean_year_string)

# STEP 1: Add the 'new' column to each dataset
df_used['new'] = 'no'
df_new['new'] = 'yes'

# STEP 2: Prepare new cars dataset
df_new['Price_clean'] = df_new['Price'].str.replace(' ', '').astype(float)

# Process climatisation for new cars
def check_auto_climatisation(clim_str):
    if pd.isna(clim_str):
        return 'No'
    clim_str = str(clim_str).lower()
    if 'automatique' in clim_str:
        return 'Yes'
    else:
        return 'no'

df_new['Climatisation_auto'] = df_new['Climatisation'].apply(check_auto_climatisation)

# Add missing columns for new cars with default values
df_new['Kilométrage'] = 0  
df_new['Mise en circulation'] = '12.2025' 

# Select and rename columns for new cars
df_new_clean = df_new[['Brand', 'Model', 'Price_clean', 'Kilométrage', 'Carrosserie', 
                      'Energie', 'Nombre de places', 'Boîte', 'Puissance fiscale', 
                      'Mise en circulation', 'Climatisation_auto', 'new']].copy()

df_new_clean = df_new_clean.rename(columns={
    'Brand': 'brand',
    'Model': 'model', 
    'Price_clean': 'price',
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Energie': 'fuel_type',
    'Nombre de places': 'seats',
    'Boîte': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year',
    'Climatisation_auto': 'climatisation'
})

# STEP 3: Prepare used cars dataset
# Select and rename columns for used cars
df_used_clean = df_used[['Spécifications - Marque', 'Spécifications - Modèle', 'Price', 
                        'Kilométrage', 'Carrosserie', 'Motorisation - Énergie',
                        'Spécifications - Nombre de places', 'Boite vitesse', 'Puissance fiscale',
                        'Mise en circulation', 'Fonctionnels - Climatisation automatique', 'new']].copy()

df_used_clean = df_used_clean.rename(columns={
    'Spécifications - Marque': 'brand',
    'Spécifications - Modèle': 'model',
    'Price': 'price', 
    'Kilométrage': 'kilometrage',
    'Carrosserie': 'body_type',
    'Motorisation - Énergie': 'fuel_type',
    'Spécifications - Nombre de places': 'seats',
    'Boite vitesse': 'transmission',
    'Puissance fiscale': 'puissance_fiscale',
    'Mise en circulation': 'registration_year',
    'Fonctionnels - Climatisation automatique': 'climatisation'
})

# STEP 4: Merge the datasets
merged_df = pd.concat([df_used_clean, df_new_clean], ignore_index=True)

# STEP 5: Display results
print(f"Final merged dataset shape: {merged_df.shape}")
print(f"Used cars: {(merged_df['new'] == 'no').sum()}")
print(f"New cars: {(merged_df['new'] == 'yes').sum()}")

# VERIFY THE FIX
print(f"\n=== REGISTRATION YEAR DISTRIBUTION (FIXED) ===")
print("Top 20 registration_year values:")
print(merged_df['registration_year'].value_counts().head(20))

print("\nFirst 15 rows of merged dataset:")
print(merged_df.head(15))

Final merged dataset shape: (2658, 12)
Used cars: 2148
New cars: 510

=== REGISTRATION YEAR DISTRIBUTION (FIXED) ===
Top 20 registration_year values:
12.2025    510
5.2022      37
11.2021     36
3.2021      34
1.2021      33
9.2020      30
9.2021      28
3.2022      28
1.2022      27
1.2020      27
6.2021      27
2.2021      26
6.2022      26
7.2020      25
7.2021      25
10.2021     24
11.2019     24
10.2020     24
12.2020     23
8.2021      22
Name: registration_year, dtype: int64

First 15 rows of merged dataset:
            brand         model     price kilometrage body_type  \
0             GWM  Haval Jolion   78000.0  113 000 km       SUV   
1            Audi            A6   75000.0  160 000 km   Berline   
2   Mercedes-Benz     GLE Coupé  460000.0   15 000 km       SUV   
3   Mercedes-Benz      Classe E  118000.0  140 000 km   Berline   
4       Chevrolet        Groove   68000.0   67 000 km       SUV   
5           Skoda         Fabia   47000.0  115 000 km  Citadine   
6        

now we've successfully kept the important columns while fixing some conflicts , next we'll be removing unnecessary rows , for example the overwhelming majority of cars have either diesel or essence, theres also a deecent nmber of orelectric and hybride as fuel type so we'll only be keeping those , also we'll be eliminating cars/brands that have very low represantation in our data well do the same with body type too , we will also remove the outliers when it comes to pricing  and year and mileage 

In [80]:
# BLOCK: DATA CLEANING - ROW FILTERING

print(f"Initial dataset shape: {merged_df.shape}")

# Create a clean copy
merged_clean = merged_df.copy()

# 1. Fuel Type Transformation - simplify categories and remove Compacte
fuel_counts = merged_clean['fuel_type'].value_counts()
print("\nFuel type distribution:")
print(fuel_counts)

# Remove the "Compacte" row first
merged_clean = merged_clean[merged_clean['fuel_type'] != 'Compacte'].copy()

# Transform fuel types into simplified categories
def simplify_fuel_type(fuel):
    fuel = str(fuel).lower()
    if 'electrique' in fuel:
        return 'electrique'
    elif 'hybride' in fuel:
        return 'hybride' 
    else:
        return fuel  # keep essence/diesel as is

merged_clean['fuel_type_simple'] = merged_clean['fuel_type'].apply(simplify_fuel_type)

# Keep all simplified fuel types (essence, diesel, hybride, electrique)
fuel_types_to_keep = ['essence', 'diesel', 'hybride', 'electrique']
merged_clean = merged_clean[merged_clean['fuel_type_simple'].isin(fuel_types_to_keep)].copy()
print(f"After fuel filtering: {merged_clean.shape}")

print("\nSimplified fuel type distribution:")
print(merged_clean['fuel_type_simple'].value_counts())

# 2. Brand Filtering - remove brands with low representation
brand_counts = merged_clean['brand'].value_counts()
print(f"\nBrand counts:")
print(brand_counts)

# Keep brands with at least 10 cars
brands_to_keep = brand_counts[brand_counts >= 10].index
merged_clean = merged_clean[merged_clean['brand'].isin(brands_to_keep)].copy()
print(f"After brand filtering: {merged_clean.shape}")

# 3. Body Type Filtering - remove rare body types
body_counts = merged_clean['body_type'].value_counts()
print(f"\nBody type counts:")
print(body_counts)

# Keep body types with at least 5 cars
bodies_to_keep = body_counts[body_counts >= 30].index
merged_clean = merged_clean[merged_clean['body_type'].isin(bodies_to_keep)].copy()
print(f"After body type filtering: {merged_clean.shape}")

# 4. Price Outlier Removal
price_stats = merged_clean['price'].describe()
print(f"\nPrice statistics before:")
print(price_stats)

# Remove extreme price outliers (bottom 1% and top 1%)
Q1_price = merged_clean['price'].quantile(0.01)
Q3_price = merged_clean['price'].quantile(0.99)
merged_clean = merged_clean[(merged_clean['price'] >= Q1_price) & (merged_clean['price'] <= Q3_price)].copy()
print(f"After price filtering: {merged_clean.shape}")


Initial dataset shape: (2658, 12)

Fuel type distribution:
Essence                           1707
Diesel                             557
Hybride rechargeable essence       127
Electrique                          95
Hybride léger essence               60
Essence | Hybride rechargeable      34
Essence | Hybride léger             25
Essence | Hybride                   23
Hybride essence                     14
Hybride léger diesel                10
Hybride rechargeable diesel          3
Diesel | Hybride léger               2
Compacte                             1
Name: fuel_type, dtype: int64
After fuel filtering: (2657, 13)

Simplified fuel type distribution:
essence       1707
diesel         557
hybride        298
electrique      95
Name: fuel_type_simple, dtype: int64

Brand counts:
Mercedes-Benz    397
Volkswagen       204
KIA              166
BMW              151
Peugeot          141
                ... 
Lexus              1
BAIC YX            1
Tata               1
Chrysler          

In [81]:
# 5. Year Filtering - CLEAN VERSION
print(f"=== YEAR FILTERING ===")

# Extract year from registration_year
def extract_year_fixed(year_str):
    if pd.isna(year_str):
        return None
    year_str = str(year_str).strip()
    if '.' in year_str:
        parts = year_str.split('.')
        if len(parts) == 2:
            year_part = parts[1]
            try:
                year = int(year_part)
                if 1900 <= year <= 2030:
                    return year
            except:
                return None
    return None

merged_clean['year_extracted'] = merged_clean['registration_year'].apply(extract_year_fixed)

# Remove cars outside 2007-2025 range and null years
current_year = 2025
initial_count = len(merged_clean)
merged_clean = merged_clean[
    (merged_clean['year_extracted'] >= 2007) & 
    (merged_clean['year_extracted'] <= current_year) &
    (merged_clean['year_extracted'].notnull())
].copy()

print(f"Rows removed by year filtering: {initial_count - len(merged_clean)}")
print(f"Final dataset shape: {merged_clean.shape}")

=== YEAR FILTERING ===
Rows removed by year filtering: 35
Final dataset shape: (2336, 14)


In [84]:
# 6. Mileage Outlier Removal (for used cars only, keep new cars with 0 km)

# Convert kilometrage properly - remove 'km' and spaces, then convert to float
merged_clean['kilometrage'] = merged_clean['kilometrage'].astype(str).str.replace(' km', '').str.replace(' ', '').astype(float)

# Remove unrealistic mileages (e.g., over 500,000 km) but keep new cars (0 km)
merged_clean = merged_clean[(merged_clean['kilometrage'] <= 250000) | (merged_clean['new'] == 'yes')].copy()
print(f"After mileage filtering: {merged_clean.shape}")

# Final summary
print(f"\n=== CLEANING SUMMARY ===")
print(f"Initial rows: {len(merged_df)}")
print(f"Final rows: {len(merged_clean)}")
print(f"Rows removed: {len(merged_df) - len(merged_clean)}")
print(f"Removal percentage: {((len(merged_df) - len(merged_clean)) / len(merged_df) * 100):.1f}%")



After mileage filtering: (2230, 14)

=== CLEANING SUMMARY ===
Initial rows: 2658
Final rows: 2230
Rows removed: 428
Removal percentage: 16.1%
